# Data collected 20/10/2026 21:13

In [1]:
import datetime as dt

import pandas as pd
import yfinance as yf
from yaml import safe_dump, safe_load

from rfm_sp500 import constants

print(f"Last run: {dt.datetime.now()}")

Last run: 2026-11-24 00:12:32.353771


In [2]:
metadata_file_path = constants.RAW_DATA_DIR / "metadata.yaml"

if metadata_file_path.exists():
    with open(metadata_file_path, "r") as f:
        metadata_dict = safe_load(f)
else:
    metadata_dict = {"yfinance version": yf.__version__}

url = constants.TICKER_URL

tickers_path = constants.RAW_DATA_DIR / "sp500_tickers.csv.gz"

if not tickers_path.exists():
    tickers_df = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})[0]
    tickers_df["Symbol"] = tickers_df["Symbol"].str.replace(".", "-")

    tickers_df.to_csv(tickers_path, index=False)
    print(f"Saved S&P 500 tickers to {tickers_path}")
    print(f"Number of tickers: {len(tickers_df)}")
    metadata_dict.update({"sp500_tickers.csv.gz": {"last_updated": str(dt.datetime.now())}})


else:
    tickers_df = pd.read_csv(tickers_path)
    print(f"Loaded S&P 500 tickers from {tickers_path}")
    print(f"Number of tickers: {len(tickers_df)}")

Loaded S&P 500 tickers from D:\dev\crm-rfm-sp500\data\raw\sp500_tickers.csv.gz
Number of tickers: 503


In [3]:
with open(constants.CONFIG_FILE, "r") as f:
    config = safe_load(f)

start_date = config["start_date"]
end_date_exclusive = config["end_date_exclusive"]

tickers = tickers_df["Symbol"].tolist()

In [4]:
save_path_sp = constants.RAW_DATA_DIR / f"sp500_data_{start_date}_{end_date_exclusive}.csv.gz"

if not save_path_sp.exists():
    sp = yf.download(
        tickers, start=start_date, end=end_date_exclusive, auto_adjust=False, actions=True
    )

    sp = (
        sp.stack(level="Ticker", future_stack=True)
        .reset_index()
        .sort_values(by=["Ticker", "Date"])
        .reset_index(drop=True)
    )
    sp.columns.name = None

    sp.to_csv(save_path_sp, index=False)
    print(f"Saved S&P 500 data to {save_path_sp}")

    metadata_dict.update(
        {
            f"sp500_data_{start_date}_{end_date_exclusive}.csv.gz": {
                "last_updated": str(dt.datetime.now())
            }
        }
    )

else:
    sp = pd.read_csv(save_path_sp, parse_dates=["Date"])
    print(f"Loaded S&P 500 data from {save_path_sp}")

sp.head()

Loaded S&P 500 data from D:\dev\crm-rfm-sp500\data\raw\sp500_data_2021-01-01_2025-01-01.csv.gz


,Date,Ticker,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,2021-01-04,A,114.009102,118.639999,0.194,120.089996,117.110001,118.940002,0.0,2030700.0
1,2021-01-05,A,114.941277,119.610001,0.000,120.320000,118.070000,118.480003,0.0,2344000.0
2,2021-01-06,A,118.093231,122.889999,0.000,123.879997,118.209999,118.690002,0.0,2428500.0
3,2021-01-07,A,121.235580,126.160004,0.000,127.480003,124.110001,124.330002,0.0,1775900.0
4,2021-01-08,A,122.100464,127.059998,0.000,127.639999,125.849998,126.489998,0.0,1577200.0


In [5]:
no_data = sp.groupby("Ticker")["Adj Close"].apply(lambda s: s.isna().all())
print(f"Tickers with no data: {no_data[no_data].index.tolist()}")

exclusion_dict = {ticker: "No data available" for ticker in no_data[no_data].index}
exclusion_file_path = (
    constants.PROCESSED_DATA_DIR / f"exclusion_dict_{start_date}_{end_date_exclusive}.yaml"
)

with open(exclusion_file_path, "w") as f:
    safe_dump(exclusion_dict, f)

print(f"Saved exclusions to {exclusion_file_path}")

Tickers with no data: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']
Saved exclusions to D:\dev\crm-rfm-sp500\data\processed\exclusion_dict_2021-01-01_2025-01-01.yaml


In [6]:
benchmark = config["benchmark"]

save_path_benchmark = (
    constants.RAW_DATA_DIR / f"benchmark_data_{start_date}_{end_date_exclusive}.csv.gz"
)

if not save_path_benchmark.exists():
    benchmark_df = yf.download(
        benchmark,
        start=start_date,
        end=end_date_exclusive,
        auto_adjust=False,
        actions=True,
        multi_level_index=False,
    ).reset_index()

    benchmark_df.to_csv(save_path_benchmark, index=False)
    print(f"Saved {benchmark} index data to {save_path_benchmark}")

    metadata_dict.update(
        {
            f"benchmark_data_{start_date}_{end_date_exclusive}.csv.gz": {
                "last_updated": str(dt.datetime.now())
            }
        }
    )

else:
    benchmark_df = pd.read_csv(save_path_benchmark, parse_dates=["Date"])
    print(f"Loaded {benchmark} index data from {save_path_benchmark}")

benchmark_df.head()

Loaded ^GSPC index data from D:\dev\crm-rfm-sp500\data\raw\benchmark_data_2021-01-01_2025-01-01.csv.gz


,Date,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,2021-01-04,3700.649902,3700.649902,0.0,3769.989990,3662.709961,3764.610107,0.0,5015000000
1,2021-01-05,3726.860107,3726.860107,0.0,3737.830078,3695.070068,3698.020020,0.0,4591020000
2,2021-01-06,3748.139893,3748.139893,0.0,3783.040039,3705.340088,3712.199951,0.0,6064110000
3,2021-01-07,3803.790039,3803.790039,0.0,3811.550049,3764.709961,3764.709961,0.0,5099160000
4,2021-01-08,3824.679932,3824.679932,0.0,3826.689941,3783.600098,3815.050049,0.0,4773040000


In [7]:
with open(metadata_file_path, "w") as f:
    safe_dump(metadata_dict, f, sort_keys=False)

print(f"Saved metadata to {metadata_file_path}")

Saved metadata to D:\dev\crm-rfm-sp500\data\raw\metadata.yaml
